esse código baixa todas as bibliotecas que serão utilizadas no experimento

In [ ]:
!pip install -r requirements.txt

# Realização dos testes na base nova

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,  recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

Realizando o carregamento das bases de falsos negativos e verdadeiros negativos criada no arquivo 'criacao_base_fn.ipynb'

In [ ]:
fn_base = pd.read_csv(r'base_fn_cg.csv')
tn_base = pd.read_csv(r'base_tn_cg.csv')

setando para os valores de y para o modelo

fn_base que são nossos nomes que foram classificados como palabras recebem o rótulo '1'  
tn_base que são nossas palavras que foram classificados corretamente como palabras recebem o rótulo '0'

In [ ]:
fn_base['y_true'] = 1
tn_base['y_true'] = 0

concatenação das duas bases (base_fn_cg.csv e base_tn_cg.csv)

In [ ]:
base_final = pd.concat([fn_base, tn_base])

In [ ]:
base_final = base_final.reset_index(drop=True)

In [ ]:
y = base_final['y_true']
X = base_final.drop(['y_true'], axis=1)

# Passo 1 - Realização da divisão da base para treino e teste - (30% para treino 70% para teste)

aqui é realizada a divisão da base para treino e teste

estabelecendo o critério de divisão 30-70

30% da base sendo utilizada para teste  
70% da base sendo utilizada para treinamento

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
def tabela_divisao(y_train, y_test):
    treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
    teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
    df_dist = pd.concat([treino, teste], axis=1)
    df_dist['Total'] = df_dist.sum(axis=1)
    pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
    return df_dist.T

In [ ]:
tabela_divisao(y_train, y_test)

In [ ]:
y_predictions = {}

# Passo 2 - Realização do treino e teste dos modelos utilizando os modelos presentes no artigo.

aqui é realizado o carregamento dos algoritmos de classificação e iniciado o processo de treinamento e teste dos modelos

# Support Vector Machine

In [ ]:
from sklearn.svm import SVC

clf = SVC(gamma='auto', random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_predictions['SVM'] = clf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['SVM'])

# Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

In [ ]:
y_predictions['Random Forest'] = rf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Random Forest'])

## explicabilidade random forest

In [ ]:
import lime
import numpy as np
import lime.lime_tabular

In [ ]:
predict_fn_rf = lambda x: rf.predict_proba(x).astype(float)
X = X_train.values
# explainer = lime.lime_tabular.LimeTabularExplainer(X,feature_names = X_train.columns, discretize_continuous=True)
explainer = lime.lime_tabular.LimeTabularExplainer(X,feature_names = X_train.columns)

In [ ]:
X_test

In [ ]:
choosen_instance = X_test.values[0]
exp = explainer.explain_instance(choosen_instance, predict_fn_rf,num_features=10)
exp.show_in_notebook(show_all=False)

In [ ]:
choosen_instance = X_test.values[4]
exp = explainer.explain_instance(choosen_instance, predict_fn_rf,num_features=10)
exp.show_in_notebook(show_all=False)

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_predictions['Logistic Regression'] = model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Logistic Regression'])

# Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

In [ ]:
y_predictions['Gradient Boosting'] = gb.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Gradient Boosting'])

# Confusion Matrix

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 6))

for model_name, y_pred, ax in zip(y_predictions.keys(), y_predictions.values(), axs.flat):
    cm = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred), display_labels=clf.classes_, )
    cm.plot(ax=ax, cmap='binary', values_format='d')
    ax.grid(False)
    ax.set_title(model_name)
plt.tight_layout()
# plt.savefig('imagens/matriz_confusao.png', bbox_inches='tight', dpi = 300)

In [ ]:
dfs = {}
for model_name, y_pred in y_predictions.items():
    report = {}
    report['accuracy'] = accuracy_score(y_test, y_pred)
    report['precision'] = precision_score(y_test, y_pred)
    report['recall'] = recall_score(y_test, y_pred)
    report['f1-score'] = f1_score(y_test, y_pred)
    dfs[model_name] = pd.DataFrame(report, index=[model_name])

# Metrics - ACCURACY, PRECISION, RECALL, F1-SCORE

In [ ]:
concatenated_df = pd.concat(dfs.values())
concatenated_df

In [ ]:
plt.style.use('bmh')

# Comparação métricas dos algoritmos

In [ ]:
concatenated_df.T.plot.bar(figsize=(7, 3), ylim=(0, 1.0), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], grid=1).legend(bbox_to_anchor=(1, 1))
plt.xticks(rotation=0)
plt.tight_layout()

# Validação cruzada

In [ ]:
def validacao_cruzada(X_train, y_train, models):
    cv_results= {}
    scores = {}

    for model_name, model in models.items():
        score = cross_val_score(estimator=model, X=X_train, y=y_train, cv=10)
        media = score.mean()
        desvio = score.std()
        scores[model_name] = score
        cv_results[model_name] = {'Validação Cruzada': "%.5f" % media, 'Desvio Padrão': "%.5f" % desvio}
        
    return cv_results, scores

In [ ]:
cv_results, scores = validacao_cruzada(X_train, y_train,{
    'SVC': SVC(),
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Gradient Boosting': GradientBoostingClassifier()
})

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores.values(), labels=scores.keys(), patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

In [ ]:
scores